In [9]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Create a view to the pothole data
RAW_DATA = 'data/raw/dot_pothole_repairs.parquet'
con.sql(f"""
    CREATE OR REPLACE VIEW raw_dot AS
    SELECT * FROM '{RAW_DATA}'
""")

In [10]:
con.sql("DESCRIBE raw_dot")

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ defnum       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ initby       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ housenum     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ oft          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ onfacename   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ onprimname   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ frmprimnam   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ toprimname   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ boro         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ source       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rptd

In [20]:
con.sql("""SELECT COUNT(*) FROM raw_dot WHERE centroid_lon IS NULL""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

In [ ]:
# Check for duplicates (shared coordinates and date)
con.sql("""
    SELECT 
        SUM(n_in_group - 1) AS rows_to_drop
    FROM (
        SELECT centroid_lat, centroid_lon, DATE(rptclosed), COUNT(*) AS n_in_group
        FROM raw_dot
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""")

┌──────────────┐
│ rows_to_drop │
│    int128    │
├──────────────┤
│          956 │
└──────────────┘

In [ ]:
# Build the cleaning SQL
msg = """
WITH source AS (
    SELECT
        onfacename,
        boro,
        source,
        rptdate,
        rptclosed,
        centroid_lat,
        centroid_lon
    FROM 'data/raw/dot_pothole_repairs.parquet'
), repairs_enriched AS (
    SELECT *,
    -- Spell out borough to match 311
    CASE
        WHEN boro = 'M' THEN 'MANHATTAN'
        WHEN boro = 'X' THEN 'BRONX'
        WHEN boro = 'B' THEN 'BROOKLYN'
        WHEN boro = 'Q' THEN 'QUEENS'
        WHEN boro = 'S' THEN 'STATEN ISLAND'
    END AS borough,
    -- Month closed
    DATE_TRUNC('month', rptdate) AS report_month,
    -- Time to close
    DATE_DIFF('day', rptdate, rptclosed) AS days_to_close
    FROM source
), repairs_ranked AS (
    SELECT *,
    ROW_NUMBER() OVER (
        PARTITION BY centroid_lat, centroid_lon, DATE(rptclosed)
        ORDER BY rptclosed ASC -- keeping format of 311 cleaning
    ) AS row_num
    FROM repairs_enriched
), repairs_deduped AS (
    SELECT *
    FROM repairs_ranked
    WHERE row_num = 1
) SELECT * EXCLUDE (boro, row_num) FROM repairs_deduped
"""
con.sql(msg).df()

,onfacename,source,rptdate,rptclosed,centroid_lat,centroid_lon,borough,month_closed,days_to_close
0,ROCKAWAY PY,YRD,2004-02-08,2023-01-27,40.630622,-73.886149,BROOKLYN,2023-01-01,6928
1,58 ST,CTZ,2011-08-02,2023-08-26,40.643906,-74.020439,BROOKLYN,2023-08-01,4407
2,SLOSSON AV,CTZ,2015-09-17,2023-05-11,40.610573,-74.116729,STATEN ISLAND,2023-05-01,2793
3,23 ST,HIQ,2020-07-10,2023-07-11,40.774658,-73.922040,QUEENS,2023-07-01,1096
4,CATHERINE ST,CTZ,2022-02-22,2023-04-12,40.713769,-73.997402,MANHATTAN,2023-04-01,414
...,...,...,...,...,...,...,...,...,...
32371,STARLING AV,CTZ,2024-12-31,2024-12-31,40.835974,-73.855492,BRONX,2024-12-01,0
32372,WESTCHESTER AV,YRD,2024-12-31,2024-12-31,40.831492,-73.867704,BRONX,2024-12-01,0
32373,WHITE PLAINS RD,YRD,2024-12-31,2024-12-31,40.862279,-73.867454,BRONX,2024-12-01,0
32374,E 32 ST,YRD,2024-12-31,2024-12-31,40.746138,-73.983088,MANHATTAN,2024-12-01,0


In [13]:
con.sql("""
    SELECT boro, COUNT(*) as n_rows
    FROM raw_dot
    GROUP BY boro,
    ORDER BY n_rows DESC
""")

┌─────────┬────────┐
│  boro   │ n_rows │
│ varchar │ int64  │
├─────────┼────────┤
│ Q       │  12022 │
│ B       │   8502 │
│ M       │   4843 │
│ X       │   4022 │
│ S       │   3943 │
└─────────┴────────┘

In [42]:
msg = open('sql/02_clean_dot_repairs.sql').read()

con.sql(msg).df()

,onfacename,source,rptdate,rptclosed,centroid_lat,centroid_lon,borough,report_month,days_to_close
0,CROPSEY AV,YRD,2023-01-01,2023-01-02,40.599483,-74.001769,BROOKLYN,2023-01-01,1
1,KINGSLAND AV,YRD,2023-01-01,2023-01-02,40.721780,-73.941015,BROOKLYN,2023-01-01,1
2,AV W,YRD,2023-01-01,2023-01-02,40.592248,-73.974401,BROOKLYN,2023-01-01,1
3,MADISON AV,CTZ,2023-01-01,2023-01-03,40.748098,-73.982715,MANHATTAN,2023-01-01,2
4,INDEPENDENCE AV,CTZ,2023-01-01,2023-01-05,40.884448,-73.916535,BRONX,2023-01-01,4
...,...,...,...,...,...,...,...,...,...
32382,ATLANTIC AV,CTZ,2024-12-31,2025-01-06,40.689151,-73.845469,QUEENS,2024-12-01,6
32383,53 ST,CTZ,2024-12-31,2025-01-03,40.630792,-73.991146,BROOKLYN,2024-12-01,3
32384,3 AV,YRD,2024-12-31,2025-01-01,40.660839,-74.000892,BROOKLYN,2024-12-01,1
32385,HYLAN BL,CTZ,2024-12-31,2025-01-03,40.527402,-74.166063,STATEN ISLAND,2024-12-01,3
